# SGCRL Towers of Hanoi Visualization

In [ ]:
# Import required libraries
import itertools
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.animation as animation
from IPython.display import HTML, display

# Import Hanoi environment
import sys
sys.path.append('/u/gliu6/explaining_sgcrl')
from hanoi_env import HanoiEnv

# Set up matplotlib for inline display
plt.ion()
%matplotlib inline


## Environment Setup


In [ ]:
# Configure the Towers of Hanoi puzzle
NUM_DISKS = 3 
env = HanoiEnv()
env.set_env_parameters(num_disks=NUM_DISKS, env_noise=0, verbose=False)

# Generate all possible states (each disk can be on peg 0, 1, or 2)
all_disk_positions = list(range(3))
all_states = list(itertools.product(all_disk_positions, repeat=NUM_DISKS))

# Create state-to-index mapping for efficient lookup
state_to_idx = {state: idx for idx, state in enumerate(all_states)}
idx_to_state = {idx: state for state, idx in state_to_idx.items()}

# Environment parameters
NUM_STATES = len(all_states)
NUM_ACTIONS = 6  # 6 possible moves between pegs (0→1, 0→2, 1→0, 1→2, 2→0, 2→1)
START_STATE = state_to_idx[(0,) * NUM_DISKS]  # All disks start on peg 0
GOAL_STATE = state_to_idx[(2,) * NUM_DISKS]   # All disks should end on peg 2

print(f"Hanoi Environment Setup:")
print(f"   Number of disks: {NUM_DISKS}")
print(f"   Number of states: {NUM_STATES}")
print(f"   Number of actions: {NUM_ACTIONS}")
print(f"   Start state: {START_STATE} -> {idx_to_state[START_STATE]}")
print(f"   Goal state: {GOAL_STATE} -> {idx_to_state[GOAL_STATE]}")

def step(state_idx: int, action: int) -> int:
    """
    Deterministic step function for the Hanoi puzzle.
    
    Args:
        state_idx: Current state index
        action: Action to take (0-5 for moves between pegs)
    
    Returns:
        Next state index after taking the action
    """
    current_state_tuple = idx_to_state[state_idx]
    
    # Reset environment to current state
    env.current_state = current_state_tuple
    env.done = False
    
    # Take the action
    next_state_tuple, reward, done, info = env.step(action)
    
    # Return the index of the next state
    return state_to_idx[next_state_tuple]



## SGCRL Agent Implementation

In [ ]:
class SGCRLAgent:
    """
    State-Goal Contrastive Reinforcement Learning Agent for Hanoi
    """
    
    def __init__(self, nState, nAction, rep_dim=16, episodes_per_upd=5, 
                 lr_psi=1e-3, replay_capacity=1000, max_steps=100, batch_size=128,
                 gamma=0.99, entropy_coeff=0.1, max_episodes=50000, plot_freq=100):
        """
        Initialize SGCRL agent for Hanoi
        
        Args:
            nState: number of states
            nAction: number of actions
            rep_dim: representation dimension
            episodes_per_upd: episodes between representation updates
            lr_psi: learning rate for psi
            replay_capacity: maximum replay buffer size
            max_steps: maximum steps per episode
            batch_size: batch size for training
            gamma: discount factor for trajectory sampling
            entropy_coeff: entropy coefficient for action selection
            max_episodes: maximum training episodes
            plot_freq: frequency for plotting (episodes)
        """
        # Store basic parameters
        self.nState = nState
        self.nAction = nAction
        self.rep_dim = rep_dim
        self.episodes_per_upd = episodes_per_upd
        self.lr_psi = lr_psi
        self.replay_capacity = replay_capacity
        self.max_steps = max_steps
        self.batch_size = batch_size
        self.gamma = gamma
        self.entropy_coeff = entropy_coeff
        self.max_episodes = max_episodes
        self.plot_freq = plot_freq
        self.norm = True
        
        # Set up goal
        self.goal = GOAL_STATE
        
        # Initialize representation
        self.psi = np.empty((nState, rep_dim))
        
        # Initialize goal embedding
        self.psi_goal = np.random.randn(rep_dim) * 0.1
        self.psi[self.goal] = self.psi_goal
        if self.norm:
            psi_norm = np.linalg.norm(self.psi[self.goal]) + 1e-8
            self.psi[self.goal] /= psi_norm
            self.psi_goal /= psi_norm
        
        # Initialize other states
        for s in range(nState):
            if s != self.goal:
                self.psi[s] = self.psi_goal + np.random.randn(rep_dim) * 0.1
        
        # Normalize psi to unit vectors
        if self.norm:
            psi_norms = np.linalg.norm(self.psi, axis=1, keepdims=True) + 1e-8
            self.psi /= psi_norms
        
        # Replay buffer and tracking
        self.replay = []
        self.visited_states = set()
        self.visited_counts = []
        self.loss_history = []
        self.success_list = []
        self.eval_success_list = []
        self.similarity_history = []  # Store similarity plots for animation

    def select_action(self, s: int, g: int) -> int:
        """
        Softmax action selection based on ψ(s') · ψ(g) similarity,
        scaled by 1 / entropy_coeff (i.e., inverse temperature)
        """
        goal_vec = self.psi[g]

        similarities = []
        for a in range(self.nAction):
            ns = step(s, a)
            sim = self.psi[ns] @ goal_vec
            similarities.append(sim)

        # Apply entropy regularization (inverse temperature)
        logits = np.array(similarities)
        inverse_temp = 1.0 / self.entropy_coeff
        logits *= inverse_temp

        # Softmax over scaled logits
        exp_logits = np.exp(logits - np.max(logits))
        probs = exp_logits / np.sum(exp_logits)

        return np.random.choice(self.nAction, p=probs)

    def eval_action(self, s: int, g: int) -> int:
        """
        Deterministic evaluation policy: chooses the action with max cosine similarity
        between ψ(s') and ψ(g). No exploration or sampling.
        """
        goal_vec = self.psi[g]

        best_a = 0
        best_val = -np.inf

        for a in range(self.nAction):
            ns = step(s, a)
            ns_vec = self.psi[ns]
            val = ns_vec @ goal_vec
            if val > best_val:
                best_val = val
                best_a = a

        return best_a

    def collect_episode(self):
        """Generate one trajectory and push into replay buffer."""
        step_success = []
        traj = [START_STATE]
        for _ in range(self.max_steps):
            a = self.select_action(traj[-1], self.goal)
            ns = step(traj[-1], a)
            traj.append(ns)
            success = (ns == self.goal)
            step_success.append(1 if success else 0)

        self.replay.append(traj)
        if len(self.replay) > self.replay_capacity:
            self.replay.pop(0)

        return step_success

    def run_eval_episode(self):
        """Generate one evaluation trajectory."""
        success = 0
        traj = [START_STATE]
        for _ in range(self.max_steps):
            a = self.eval_action(traj[-1], self.goal)
            ns = step(traj[-1], a)
            traj.append(ns)
            success = (ns == self.goal)

        return success, traj

    def update_representations(self) -> float:
        """
        Vectorised contrastive update
        """
        if len(self.replay) < 2:
            return 0.0

        traj_ids = np.random.choice(len(self.replay), self.batch_size, replace=True)
        s_list, sp_list = [], []
        for idx in traj_ids:
            traj = self.replay[idx]
            if len(traj) < 2:
                continue
            i = np.random.randint(0, len(traj) - 1)
            remaining = len(traj) - i
            w = self.gamma ** np.arange(remaining)
            w /= w.sum()
            j = i + np.random.choice(remaining, p=w)

            s_list.append(traj[i])
            sp_list.append(traj[j])

        if not s_list:
            return 0.0

        s_batch = np.asarray(s_list, dtype=np.int32)
        sp_batch = np.asarray(sp_list, dtype=np.int32)
        B = len(s_batch)

        # Gather current embeddings
        psi_s = self.psi[s_batch]
        psi_p = self.psi[sp_batch]

        # Column-wise soft-max probabilities P
        dots = psi_s @ psi_p.T
        dots -= dots.max(axis=0, keepdims=True)
        exp_logits = np.exp(dots)
        P = exp_logits / exp_logits.sum(axis=0, keepdims=True)

        diag_P = np.diag(P)
        nll = -np.mean(np.log(diag_P + 1e-12))

        # Anchor-state updates Δψ(s_j)
        coeff = np.eye(B) - P
        anchor_update = self.lr_psi * (coeff @ psi_p)
        np.add.at(self.psi, s_batch, anchor_update)

        # Positive-state updates Δψ(sp_k)
        expected_anchor = (P.T @ psi_s)
        pos_update = self.lr_psi * (psi_s - expected_anchor)
        np.add.at(self.psi, sp_batch, pos_update)
        
        # Normalize ψ to unit vectors (L2 norm)
        if self.norm:
            psi_norms = np.linalg.norm(self.psi, axis=1, keepdims=True) + 1e-8
            self.psi /= psi_norms

        return nll
    
    def create_similarity_plot(self, episode, eval_traj):
        """Create similarity and visitation plot for animation"""
        goal_vec = self.psi[self.goal]
        goal_norm = np.linalg.norm(goal_vec) + 1e-8
        sim = (self.psi @ goal_vec) / (np.linalg.norm(self.psi, axis=1) * goal_norm + 1e-8)

        # Get the last 4 training trajectories from the replay buffer
        # These are the most recent training trajectories
        train_trajs = self.replay[-4:] if len(self.replay) >= 4 else self.replay
        
        # The eval trajectory will ALWAYS be shown as the LAST trajectory in the visitation plot
        # This is the trajectory that will be animated on the right side
        trajs_to_analyze = train_trajs + [eval_traj]
        num_trajs = len(trajs_to_analyze)
        
        # Count visits per state per trajectory
        visit_counts = np.zeros((self.nState, num_trajs))  # states x trajectories
        for i, traj in enumerate(trajs_to_analyze):
            for state in traj:
                visit_counts[state, i] += 1
        
        self.similarity_history.append((episode, sim, visit_counts, num_trajs, eval_traj))
        
        return None

    def train(self):
        """Main training loop"""
        for ep in tqdm(range(0, self.max_episodes), desc="Training episodes"):
            ep_success = self.collect_episode()
            self.success_list.append(np.mean(ep_success))
            
            for state in self.replay[-1]:
                self.visited_states.add(state)
            self.visited_counts.append(len(self.visited_states))

            # Perform SGD update every N episodes
            if ep % self.episodes_per_upd == 0:
                loss = self.update_representations()
                self.loss_history.append(loss)

            # Visualize and create plots
            if (ep % self.plot_freq == 0 or ep == 0):
                eval_ep_success, eval_traj = self.run_eval_episode()
                self.eval_success_list.append(eval_ep_success)
                
                # Store similarity and visitation data (don't display during training)
                self.create_similarity_plot(ep, eval_traj)


    def plot_success_rate(self, window=100):
        """Plot evaluation success rate over training episodes with smoothing"""
        plt.figure(figsize=(8, 4))
        
        # Convert success list to binary (0 or 1)
        successes = np.array([1 if r > 0 else 0 for r in self.success_list])
        
        # Compute moving average
        smoothed = []
        for i in range(len(successes)):
            start_idx = max(0, i - window + 1)
            smoothed.append(np.mean(successes[start_idx:i+1]))
        
        episodes = [i for i in range(len(self.success_list))]
        plt.plot(episodes, smoothed, color='tab:blue', linewidth=2)
        plt.xlabel('Training Episodes', fontsize=12)
        plt.ylabel('Success Rate', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        return None


## Configuration and Training

In [ ]:
config = {
    "rep_dim": 16,
    "episodes_per_upd": 5,
    "lr_psi": 1e-3,
    "replay_capacity": 1000,
    "max_steps": 20,
    "batch_size": 128,
    "max_episodes": 2000,
    "gamma": 0.99,
    "entropy_coeff": 0.1,
    "plot_freq": 100,
    "seed": 42
}

sgcrl_agent = SGCRLAgent(
    nState=NUM_STATES, 
    nAction=NUM_ACTIONS, 
    rep_dim=config["rep_dim"],
    episodes_per_upd=config["episodes_per_upd"],
    lr_psi=config["lr_psi"],
    replay_capacity=config["replay_capacity"],
    max_steps=config["max_steps"],
    batch_size=config["batch_size"],
    gamma=config["gamma"],
    entropy_coeff=config["entropy_coeff"],
    max_episodes=config["max_episodes"],
    plot_freq=config["plot_freq"]
)

sgcrl_agent.train()
sgcrl_agent.plot_success_rate()


## Visualization of Training
### In the animation, we visualize four (actor) trajectories used for training, collected by the SGCRL policy with entropy regularization. We also visualize one evaluation trajectory, collected without entropy regularization. The success rate curves are computed using success of the actor trajectories.

In [ ]:
def render_hanoi_step(ax, state_tuple, step_num, total_steps, episode_num, is_goal):
    """Render a single step of the Hanoi trajectory"""
    ax.clear()
    ax.axis('off')
    
    # Set up environment to this state
    env.current_state = state_tuple
    env.done = False
    
    # Get disks on each peg
    peg_0_disks = sorted(env.disks_on_peg(0))
    peg_1_disks = sorted(env.disks_on_peg(1))
    peg_2_disks = sorted(env.disks_on_peg(2))
    
    max_height = NUM_DISKS
    
    # Build tower display
    tower_lines = []
    for height in range(max_height - 1, -1, -1):
        line_parts = []
        for peg_disks in [peg_0_disks, peg_1_disks, peg_2_disks]:
            bottom_height = len(peg_disks) - 1 - height
            if bottom_height >= 0:
                disk_num = peg_disks[bottom_height]
                disk_width = 2 * (disk_num + 1)
                disk = ('=' * disk_width).center(8)
            else:
                disk = '||'.center(8)
            line_parts.append(disk)
        tower_lines.append('  '.join(line_parts))
    
    # Add base
    base_line = '  '.join(['=' * 8] * 3)
    tower_lines.append(base_line)
    
    # Add peg labels
    label_line = '  '.join(['Peg 0'.center(8), 'Peg 1'.center(8), 'Peg 2'.center(8)])
    tower_lines.append(label_line)
    
    # Display the tower
    tower_text = '\n'.join(tower_lines)
    
    # Determine color and background based on goal state
    text_color = 'green' if is_goal else 'black'
    bg_color = 'lightgreen' if is_goal else 'white'
    
    # Display with larger font
    ax.text(0.5, 0.6, tower_text, fontsize=20, family='monospace',
            horizontalalignment='center', verticalalignment='center',
            color=text_color, weight='bold' if is_goal else 'normal',
            bbox=dict(boxstyle='round,pad=1', facecolor=bg_color, alpha=0.8, edgecolor='black', linewidth=2))
    
    # Add step information
    step_info = f"Step {step_num} / {total_steps}"
    ax.text(0.5, 0.8, step_info, fontsize=20, 
            horizontalalignment='center', verticalalignment='top',
            weight='bold', color='black')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

def plot_similarity_and_visitation(ep_idx):
    """Plot similarity and visitation for a given episode index"""
    ax1.clear()
    ax2.clear()
    
    # Top left plot: Cosine similarity bar chart
    states = range(NUM_STATES)
    ax1.bar(states, similarities[ep_idx], color="tab:purple")
    
    ax1.set_ylabel("ψ-similarity\nto Goal", fontsize=20)
    ax1.set_title(f"Trial {episodes[ep_idx]}", fontsize=20)
    ax1.tick_params(axis='both', which='major', labelsize=16)  # Increase tick label size
    
    # Add markers for start and goal on top plot
    ax1.scatter(START_STATE, similarities[ep_idx][START_STATE], marker='o', s=200, color='lime', 
                label=f'Start ({idx_to_state[START_STATE]})', zorder=10, edgecolors='black', linewidth=2)
    ax1.scatter(GOAL_STATE, similarities[ep_idx][GOAL_STATE], marker='*', s=300, color='red', 
                label=f'Goal ({idx_to_state[GOAL_STATE]})', zorder=10, edgecolors='black', linewidth=2)
    
    # Bottom left plot: Visitation count stacked bar chart
    visit_counts = visit_counts_list[ep_idx]
    num_trajs = num_trajs_list[ep_idx]
    
    if num_trajs > 0:
        # Create stacked bar chart
        traj_colors = plt.cm.tab10(np.linspace(0, 1, num_trajs))
        bottom = np.zeros(NUM_STATES)
        
        for i in range(num_trajs):
            # The last trajectory is always the eval trajectory (the one being animated)
            if i == num_trajs - 1:
                label = f'Traj {i+1} (Eval - rendered)'
            else:
                label = f'Traj {i+1} (Train)'
            ax2.bar(states, visit_counts[:, i], bottom=bottom, 
                    color=traj_colors[i], label=label)
            bottom += visit_counts[:, i]
        
        # Get the total height of bars for start and goal states
        start_bar_height = np.sum(visit_counts[START_STATE, :])
        goal_bar_height = np.sum(visit_counts[GOAL_STATE, :])
        
        # Position markers slightly above the bars
        y_max = ax2.get_ylim()[1]
        offset = y_max * 0.02
        
        ax2.scatter(START_STATE, start_bar_height + offset, marker='o', s=200, color='lime', 
                    label='Start State', zorder=10, edgecolors='black', linewidth=2, clip_on=False)
        ax2.scatter(GOAL_STATE, goal_bar_height + offset, marker='*', s=300, color='red', 
                    label='Goal State', zorder=10, edgecolors='black', linewidth=2, clip_on=False)
    
    ax2.set_xlabel("State Index", fontsize=20)
    ax2.set_ylabel("Visitation\nCount", fontsize=20)
    ax2.tick_params(axis='both', which='major', labelsize=16) 
    ax2.legend(bbox_to_anchor=(1.5, 0.7), loc='upper center', ncol=2, fontsize=20)

# Track current episode to know when to update left plots
current_episode = [-1]  

def animate(frame_idx):
    ep_idx, step_idx = frame_data[frame_idx]
    
    if ep_idx != current_episode[0]:
        plot_similarity_and_visitation(ep_idx)
        current_episode[0] = ep_idx
    
    traj = eval_trajs[ep_idx]
    state_idx = traj[step_idx]
    state_tuple = idx_to_state[state_idx]
    is_goal = (state_tuple == idx_to_state[GOAL_STATE])
    
    render_hanoi_step(ax3, state_tuple, step_idx, len(traj)-1, episodes[ep_idx], is_goal)
    
    return ax1, ax2, ax3

In [ ]:


# Extract data for animation
episodes = [item[0] for item in sgcrl_agent.similarity_history]
similarities = [item[1] for item in sgcrl_agent.similarity_history]
visit_counts_list = [item[2] for item in sgcrl_agent.similarity_history]
num_trajs_list = [item[3] for item in sgcrl_agent.similarity_history]
eval_trajs = [item[4] for item in sgcrl_agent.similarity_history]

# Create figure with three subplots (2 on left, 1 on right)
fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(2, 2, width_ratios=[1.2, 1], hspace=0.1, wspace=0.1)
ax1 = fig.add_subplot(gs[0, 0])  # Top left: similarity
ax2 = fig.add_subplot(gs[1, 0])  # Bottom left: visitation
ax3 = fig.add_subplot(gs[:, 1])  # Right: Hanoi visualization

# Build frame mapping: each episode will have multiple frames (one per trajectory step)
frame_data = []  # List of (episode_idx, traj_step_idx)
for ep_idx in range(len(episodes)):
    traj = eval_trajs[ep_idx]
    for step_idx in range(len(traj)):
        frame_data.append((ep_idx, step_idx))

print(f"Total frames to animate: {len(frame_data)}")


anim = animation.FuncAnimation(fig, animate, frames=len(frame_data), 
                                interval=100, blit=False, repeat=True)

# Increase the embed limit and optimize animation quality
plt.rcParams['animation.embed_limit'] = 100  

# Save animation in both gif and mp4 formats
anim.save('figures/hanoi3_animation.gif', writer='pillow', fps=10)

# Display animation
display(HTML(anim.to_jshtml(fps=10, default_mode='loop')))
plt.close(fig)


# TOH with 4 disks

In [ ]:
# Configure the Towers of Hanoi puzzle with 4 disks
NUM_DISKS = 4 
env = HanoiEnv()
env.set_env_parameters(num_disks=NUM_DISKS, env_noise=0, verbose=False)

# Generate all possible states (each disk can be on peg 0, 1, or 2)
all_disk_positions = list(range(3))
all_states = list(itertools.product(all_disk_positions, repeat=NUM_DISKS))

# Create state-to-index mapping for efficient lookup
state_to_idx = {state: idx for idx, state in enumerate(all_states)}
idx_to_state = {idx: state for state, idx in state_to_idx.items()}

# Environment parameters
NUM_STATES = len(all_states)
NUM_ACTIONS = 6  # 6 possible moves between pegs (0→1, 0→2, 1→0, 1→2, 2→0, 2→1)
START_STATE = state_to_idx[(0,) * NUM_DISKS]  # All disks start on peg 0
GOAL_STATE = state_to_idx[(2,) * NUM_DISKS]   # All disks should end on peg 2

print(f"Hanoi Environment Setup (4 disks):")
print(f"   Number of disks: {NUM_DISKS}")
print(f"   Number of states: {NUM_STATES}")
print(f"   Number of actions: {NUM_ACTIONS}")
print(f"   Start state: {START_STATE} -> {idx_to_state[START_STATE]}")
print(f"   Goal state: {GOAL_STATE} -> {idx_to_state[GOAL_STATE]}")

# Train SGCRL agent
config = {
    "rep_dim": 16,
    "episodes_per_upd": 5,
    "lr_psi": 1e-3,
    "replay_capacity": 1000,
    "max_steps": 30,  
    "batch_size": 128,
    "max_episodes": 30000,
    "gamma": 0.99,
    "entropy_coeff": 0.1,
    "plot_freq": 2000,
    "seed": 42
}

sgcrl_agent = SGCRLAgent(
    nState=NUM_STATES, 
    nAction=NUM_ACTIONS, 
    rep_dim=config["rep_dim"],
    episodes_per_upd=config["episodes_per_upd"],
    lr_psi=config["lr_psi"],
    replay_capacity=config["replay_capacity"],
    max_steps=config["max_steps"],
    batch_size=config["batch_size"],
    gamma=config["gamma"],
    entropy_coeff=config["entropy_coeff"],
    max_episodes=config["max_episodes"],
    plot_freq=config["plot_freq"]
)

sgcrl_agent.train()
sgcrl_agent.plot_success_rate()


In [ ]:

# Extract data for animation
episodes = [item[0] for item in sgcrl_agent.similarity_history]
similarities = [item[1] for item in sgcrl_agent.similarity_history]
visit_counts_list = [item[2] for item in sgcrl_agent.similarity_history]
num_trajs_list = [item[3] for item in sgcrl_agent.similarity_history]
eval_trajs = [item[4] for item in sgcrl_agent.similarity_history]

# Create figure with three subplots (2 on left, 1 on right)
fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(2, 2, width_ratios=[1.2, 1], hspace=0.1, wspace=0.1)
ax1 = fig.add_subplot(gs[0, 0])  # Top left: similarity
ax2 = fig.add_subplot(gs[1, 0])  # Bottom left: visitation
ax3 = fig.add_subplot(gs[:, 1])  # Right: Hanoi visualization

# Build frame mapping: each episode will have multiple frames (one per trajectory step)
frame_data = []  # List of (episode_idx, traj_step_idx)
for ep_idx in range(len(episodes)):
    traj = eval_trajs[ep_idx]
    for step_idx in range(len(traj)):
        frame_data.append((ep_idx, step_idx))

print(f"Total frames to animate: {len(frame_data)}")


anim = animation.FuncAnimation(fig, animate, frames=len(frame_data), 
                                interval=100, blit=False, repeat=True)

# Increase the embed limit and optimize animation quality
plt.rcParams['animation.embed_limit'] = 100  # Increase limit to 100 MB

# Save animation in both gif and mp4 formats
anim.save('figures/hanoi4_animation.gif', writer='pillow', fps=10)

# Display animation
display(HTML(anim.to_jshtml(fps=10, default_mode='loop')))
plt.close(fig)  
